# Frozen Lake Experiments

This notebook uses the general `GymExperiment` runner. The runner is not tied to Frozen Lake; Frozen Lake is just the registered Gymnasium environment used here.

In [ ]:
import os
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp")

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import rl_suite.RLToolbox.algorithms as rl
import rl_suite.RLToolbox.callbacks as cb
from rl_suite.experiments import GymExperiment

%matplotlib inline

## Experiment Setup

In [ ]:
ENV = "FrozenLake-v1"
SEED = 10
EVAL_EPISODES = 100

REWARDS = {
    "sparse": (1.0, 0.0, 0.0),
    "dense": (1.0, -1.0, -0.01),
}

configs = [
    {"reward": reward_name, "gamma": gamma, "agent": {"gamma": gamma, "theta": theta}, "env": {
        "map_name": "4x4",
        "is_slippery": True,
        "reward_schedule": reward_schedule,
    }}
    for gamma in (0.9, 0.99)
    for theta in (1e-3,)
    for reward_name, reward_schedule in REWARDS.items()
]


The small training functions below adapt the existing GPI agents to the generic runner. Any other algorithm can be used the same way: provide an algorithm class/factory and, when needed, a `train` function.

In [ ]:
def make_callbacks(config):
    return cb.CallbackList([
        cb.TimerCallback(),
        cb.ConvergenceCallback(),
        cb.ValueFunctionCallback(),
        cb.PolicyCallback(),
        cb.RolloutCallback(),
    ])


def train_policy_iteration(agent, env, callbacks):
    agent.get_env_info(env)
    return agent.policy_iteration(print_num_iter=False, callbacks=callbacks)


def train_value_iteration(agent, env, callbacks):
    agent.get_env_info(env)
    return agent.value_iteration(print_num_iter=False, callbacks=callbacks)


def get_callback(row, callback_class):
    for callback in row["callbacks"].callbacks:
        if isinstance(callback, callback_class):
            return callback
    return None

## Run

In [ ]:
policy_experiment = GymExperiment(ENV, rl.gpi.deterministic, train=train_policy_iteration)
value_experiment = GymExperiment(ENV, rl.gpi.value_iteration, train=train_value_iteration)

policy_results = policy_experiment.run(
    configs=configs,
    callbacks=make_callbacks,
    eval_episodes=EVAL_EPISODES,
    seed=SEED,
)
value_results = value_experiment.run(
    configs=configs,
    callbacks=make_callbacks,
    eval_episodes=EVAL_EPISODES,
    seed=SEED,
)

for row in policy_results:
    row["algorithm"] = "Policy Iteration"
for row in value_results:
    row["algorithm"] = "Value Iteration"

results = policy_results + value_results

## Results

In [ ]:
headers = ["algorithm", "reward", "gamma", "sweeps", "success_rate", "train_ms"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]

for row in results:
    config = row["config"]
    convergence = get_callback(row, cb.ConvergenceCallback)
    timer = get_callback(row, cb.TimerCallback)
    rollouts = get_callback(row, cb.RolloutCallback)
    values = [
        row["algorithm"], config["reward"], config["gamma"], convergence.sweeps,
        f"{rollouts.success_rate:.2%}", f"{timer.elapsed * 1000:.3f}",
    ]
    lines.append("| " + " | ".join(map(str, values)) + " |")

display(Markdown("\n".join(lines)))

In [ ]:
labels = [f"{r['algorithm'][:2]} {r['config']['reward']} g={r['config']['gamma']}" for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_specs = [
    ([get_callback(r, cb.ConvergenceCallback).sweeps for r in results], "Evaluation Sweeps"),
    ([get_callback(r, cb.RolloutCallback).success_rate for r in results], "Success Rate"),
    ([get_callback(r, cb.TimerCallback).elapsed * 1000 for r in results], "Train Time (ms)"),
]

for ax, (values, title) in zip(axes, plot_specs):
    ax.bar(labels, values)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=75)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(8, 2.5 * len(results)))
for ax, row, label in zip(axes, results, labels):
    values = get_callback(row, cb.ValueFunctionCallback).final_values
    ax.plot(range(len(values)), values, marker="o")
    ax.set_title(label)
    ax.set_xlabel("State")
    ax.set_ylabel("V(s)")
    ax.grid(alpha=0.3)
fig.tight_layout()

The callbacks collect the same kinds of evidence as the reference notebook: convergence work, training time, final value functions, final policies, and rollout success/trajectory data. The algorithm-specific training functions only connect the existing PI/VI methods to those hooks.